# Week 12 — Wednesday: Time-Series Data

**DATA 202 · Calvin University**

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/12/images/bikeshare.jpg" width="480" alt="A row of red Capital Bikeshare bikes docked at a station on a Washington, D.C. sidewalk.">

Capital Bikeshare lets people in Washington, D.C. rent a bike at one station and return it at another. Every hour, someone has to decide how many bikes to move to which station, and how many mechanics to put on shift. For that they need to know how many bikes people **will** rent.

> 🎯 **How many bikes next hour? Next Monday at 8 am? And how do we test a model that predicts the future?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Two years, hour by hour |
| ~10 min | 2 · Lag features: what just happened (SLO 12B) |
| ~8 min | 3 · The clock on a circle (SLO 12B) |
| ~12 min | 4 · Test in time order: the rolling window (SLO 12B) |
| ~5 min | 5 · A toolbox for time-series data: which tool, when? (SLO 12B) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: H. Fanaee-T & J. Gama (2013), *Progress in Artificial Intelligence* 2, 113–127; Bike Sharing, UCI Machine Learning Repository, CC BY 4.0 (hours with no rentals added back as 0). Photo: MB-one, CC BY-SA 3.0, via Wikimedia Commons (F Street NW, July 2012)</sub>

---
## 1 · Two Years, Hour by Hour · ~5 min

One row = one hour, from January 1, 2011 to December 31, 2012: 17,544 hours.

* `rentals`: bikes rented that hour (what we predict)
* `temp_c`, `humidity`: the weather that hour
* `weather`: 1 = clear, 2 = mist or clouds, 3 = light rain or snow, 4 = heavy rain or snow
* `holiday`: 1 = a public holiday

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

bikes = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/bikes.csv", parse_dates=["time"])
daily = bikes.resample("D", on="time")["rentals"].sum()          # add up the 24 hours of each day
fig = px.line(daily, title="Bikes Rented per Day, 2011–2012", labels={"value": "bikes rented", "time": ""})
fig.update_layout(showlegend=False)
fig.show()
bikes.head()

In [ ]:
week = bikes[(bikes["time"] >= "2012-09-10") & (bikes["time"] < "2012-09-17")]       # Monday to Sunday
fig = px.line(week, x="time", y="rentals", title="One Week, Hour by Hour: September 10–16, 2012",
              labels={"rentals": "bikes rented per hour", "time": ""})
fig.show()

> 📄 **Handout 1 — One week.** Label the commutes and the weekend. 🗣️ What would you predict for **Monday, September 17, at 8 am**? What are you basing it on?

In [ ]:
hours = ["2012-09-10 08:00", "2012-09-16 08:00", "2012-09-17 07:00", "2012-09-17 08:00"]
bikes.set_index("time").loc[hours, ["rentals"]]

<details><summary>Answer</summary>

* **Weekdays:** two peaks, about 8 am and 5–6 pm: commuters. **Weekend:** one wide hump in the afternoon
* Across the two years: summer above winter, and **2012 above 2011** (the system kept growing). A few deep dips: storms (Hurricane Sandy, October 29, 2012: 22 bikes all day)
* Monday the 17th at 8 am: **683**. The same hour a week before: 730. The day before (Sunday) at 8 am: 128. One hour before: 492
* Your guess probably came from the **past**, picked with care: last Monday, not yesterday. That instinct is today's main tool
</details>

**Time-series data:** rows in time order, where each value depends on the ones before it. A time column alone doesn't make it time series; it's a time series when the order carries information.

---
## 2 · Lag Features: What Just Happened (SLO 12B) · ~10 min

Pull clock features out of `time`, and give a random forest (Week 11) the clock and the weather. Test it the way the bike company would use it: learn from the past (January 2011 to June 2012), then forecast the future (July to December 2012). Part 4 shows why we never split time at random.

In [ ]:
bikes["hour"] = bikes["time"].dt.hour
bikes["day_of_week"] = bikes["time"].dt.dayofweek           # 0 = Monday ... 6 = Sunday
clock_weather = ["hour", "day_of_week", "holiday", "temp_c", "humidity", "weather"]
y = bikes["rentals"]

train = bikes["time"] < "2012-07-01"          # learn from the past...
test = ~train                                 # ...and forecast the last 6 months
forest = RandomForestRegressor(random_state=1, n_jobs=-1)
forest.fit(bikes.loc[train, clock_weather], y[train])
print(f"clock + weather | test MAE: {mean_absolute_error(y[test], forest.predict(bikes.loc[test, clock_weather])):.0f} bikes per hour")

In [ ]:
results = bikes.loc[test, ["time", "rentals"]].assign(predicted=forest.predict(bikes.loc[test, clock_weather]))
by_day = results.resample("D", on="time").sum()
fig = px.line(by_day, y=["rentals", "predicted"],
              title="Clock and Weather Only: the Last 6 Months, Bikes per Day", labels={"value": "bikes per day", "time": "", "variable": ""})
fig.show()
print(f"too low on {(by_day['predicted'] < by_day['rentals']).sum()} of {len(by_day)} days | per hour, on average: "
      f"{results['predicted'].mean():.0f} predicted, {results['rentals'].mean():.0f} rented")

> 📄 **Handout 2, top — What the forest is missing.** Shade the gap between the forecast and what happened. 🗣️ Why does the forest guess too low?

<details><summary>Answer</summary>

* **MAE 82**, and too low on 170 of the 184 days: 179 bikes per hour on average, against 252 actual
* A tree predicts **averages of training rows**. It learned 2011 and early 2012; then the system grew. A forest can't predict more bikes than it has seen on a similar day
* What it's missing is what you used in Part 1: what happened **recently**, like the same hour last week
</details>

A **lag feature** is a past value of the target, moved down into the current row with `shift`.

> 📄 **Handout 2, bottom — Shift by hand.** Fill in `shift(1)` and `shift(-1)` for the first five hours of the data. 🗣️ Which row could the forest really have when it makes its prediction?

In [ ]:
bikes["rentals_1h_ago"] = bikes["rentals"].shift(1)
bikes["rentals_24h_ago"] = bikes["rentals"].shift(24)        # same hour yesterday
bikes["rentals_1wk_ago"] = bikes["rentals"].shift(168)       # same hour last week: 7 × 24 hours
bikes[["time", "rentals", "rentals_1h_ago", "rentals_24h_ago"]].head(5)

<details><summary>Answer</summary>

* `shift(1)` moves each value **down** one row: 1 hour ago. The first row has no hour before it, so pandas puts `NaN` there. A lag of 168 hours leaves the whole first week empty, so we drop those rows
* `shift(-1)` moves the **next** hour up into this row: the answer, handed over before the question. A model with it would score almost perfectly, and be useless: at prediction time, that hour hasn't happened
</details>

A **rolling** feature summarizes a recent window. Shift first, so the window ends at the hour **before** the one we predict:

In [ ]:
bikes["mean_last_24h"] = bikes["rentals"].shift(1).rolling(24).mean()     # average of the 24 hours before this one
bikes = bikes.dropna()           # the first week has no lag values

lags = ["rentals_1h_ago", "rentals_24h_ago", "rentals_1wk_ago", "mean_last_24h"]
train = bikes["time"] < "2012-07-01"
test = ~train
y = bikes["rentals"]

> 🎯 **Predict First:** No model at all, just "same as one week ago". Does it beat Part 2's forest (MAE 82)?

In [ ]:
for lag in ["rentals_1h_ago", "rentals_24h_ago", "rentals_1wk_ago"]:
    print(f"no model, just copy {lag:16} | test MAE: {mean_absolute_error(y[test], bikes.loc[test, lag]):.0f}")

forest.fit(bikes.loc[train, clock_weather + lags], y[train])
print(f"forest, clock + weather + lags | test MAE: {mean_absolute_error(y[test], forest.predict(bikes.loc[test, clock_weather + lags])):.0f}")

<details><summary>Answer</summary>

* Copying last week's value: **MAE 66**, better than Part 2's forest (82). The past carries the **level**: last week already had 2012's bigger crowds
* Copying the last hour (85) or yesterday (80) is worse: 8 am isn't like 7 am, and Monday isn't like Sunday
* **Forest with lags: MAE 33**, and on average right on the level (251 predicted, 252 actual). The clock gives the shape of the day; the lags tell where the level is now
* Always compare with a **naive forecast** (copy the past). A model that can't beat it isn't worth running
</details>

In [ ]:
results = bikes.loc[test, ["time", "rentals"]].assign(predicted=forest.predict(bikes.loc[test, clock_weather + lags]))
fig = px.line(results.resample("D", on="time").sum(), y=["rentals", "predicted"],
              title="With Lag Features: the Last 6 Months, Bikes per Day", labels={"value": "bikes per day", "time": "", "variable": ""})
fig.show()

📌 **One split:** one six-month test period, July to December 2012, including a hurricane. Another stretch of time, other numbers.

---
## 3 · The Clock on a Circle (SLO 12B) · ~8 min

So far `hour` is a number from 0 to 23, and `day_of_week` a number from 0 to 6.

> 💬 **Question:** As numbers, how far apart are 11 pm and midnight? On a clock?

To put the hour on a clock, turn it into an angle (a full turn = 24 hours), and keep two numbers: the angle's sine and cosine.

In [ ]:
bikes["hour_sin"] = np.sin(2 * np.pi * bikes["hour"] / 24)       # 2π is one full turn: 24 hours
bikes["hour_cos"] = np.cos(2 * np.pi * bikes["hour"] / 24)

clock = bikes[["hour", "hour_sin", "hour_cos"]].drop_duplicates().sort_values("hour")      # the 24 hours, once each
fig = px.scatter(clock, x="hour_sin", y="hour_cos", text="hour", title="The 24 Hours on a Circle", width=440, height=440)
fig.update_traces(textposition="top center")
fig.update_yaxes(scaleanchor="x")
fig.show()

* 11 pm and midnight: **23 apart** as numbers, **neighbors** on the circle
* Why two columns? With the sine alone, 3 am and 9 am would get the same number. The pair is different for every hour
* The same recipe works for any cycle: day of the week (`/ 7`), month (`/ 12`), day of the year (`/ 365`)

> 🎯 **Predict First:** Week 10's linear regression learns the rentals from the hour alone. What shape can it draw over the day with `hour` as a number? With `hour_sin` and `hour_cos`?

In [ ]:
noon_to_noon = list(range(12, 24)) + list(range(12))           # midnight in the middle of the plot
by_hour = bikes[train].groupby("hour")[["rentals"]].mean().loc[noon_to_noon].rename(columns={"rentals": "average"})
for name, columns in [("hour as a number", ["hour"]), ("hour on the circle", ["hour_sin", "hour_cos"])]:
    line = LinearRegression().fit(bikes.loc[train, columns], y[train])
    by_hour[name] = line.predict(clock.set_index("hour", drop=False).loc[noon_to_noon, columns])
    print(f"linear regression, {name:18} | test MAE: {mean_absolute_error(y[test], line.predict(bikes.loc[test, columns])):.0f}")

fig = px.line(by_hour, x=by_hour.index.astype(str), y=["average", "hour as a number", "hour on the circle"], markers=True,
              title="Linear Regression on the Hour Alone, Noon to Noon",
              labels={"x": "hour of the day", "value": "bikes per hour", "variable": ""})
fig.show()

> 📄 **Handout 3 — The clock on a circle.** Label lines A and B, and circle where one of them jumps. 🗣️ On a circle, Sunday sits next to Monday. Should the day of the week go on a circle too?

<details><summary>Answer</summary>

* **Line B** had `hour` as a number. A straight line in `hour` climbs all day, to 276 bikes at 11 pm, then starts over at 59 at midnight: it **jumps at midnight**, where 23 turns into 0
* **Line A**, on the circle, passes smoothly through midnight: one wave a day, low at night, high in the afternoon. Test MAE 125 instead of 155
* Neither line has the two commute peaks. More columns can draw them: the hour as **categories** (Week 9's one-hot: 24 columns, any shape), or a second wave (`np.sin(4 * np.pi * hour / 24)` and its cosine)
* **Sunday and Monday:** the circle says neighbors on the clock behave alike. True for hours (11 pm is like midnight), not for these two days: Sunday is a leisure day, Monday a commute day. A `weekend` column (or the day as categories) says what matters. When unsure, let cross-validation decide (Part 4)
</details>

Which models care? kNN measures distances (Week 6); a forest splits the hour into ranges, like "before 6 am":

In [ ]:
circle = ["hour_sin", "hour_cos", "day_of_week", "holiday", "temp_c", "humidity", "weather"]     # clock + weather, hour on the circle
knn = Pipeline([("scale", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=10))])
for name, model in [("kNN, k = 10", knn), ("random forest", forest)]:
    for label, columns in [("hour as a number", clock_weather), ("hour on the circle", circle)]:
        model.fit(bikes.loc[train, columns], y[train])
        print(f"{name:13} | {label:18} | test MAE: {mean_absolute_error(y[test], model.predict(bikes.loc[test, columns])):.0f}")

<details><summary>Answer</summary>

* **kNN: 95 → 85.** With the hour as a number, midnight is the farthest hour from 11 pm; on the circle, it's as close as 10 pm
* **Forest: 82 either way.** A tree splits the hour into ranges; it never measures how far 23 is from 0
* So: the circle (or categories) for kNN and linear models; plain numbers are fine for trees. And lags still matter more than either: Part 2's forest with lags missed by 33
</details>

---
## 4 · Test in Time Order: the Rolling Window (SLO 12B) · ~12 min

Parts 2 and 3 each scored one stretch: July to December 2012 📌. To choose features and knobs (Week 7) we need cross-validation, and it must not touch that stretch. Week 7's `KFold(shuffle=True)` deals the hours into 5 folds at random. `TimeSeriesSplit` **replays the past** instead, as a **rolling window**: learn from everything before a date, check the stretch right after it, then move the date forward.

In [ ]:
X_dev, y_dev = bikes.loc[train], y[train]           # January 2011 to June 2012: CV never sees July to December
for fold, (learn, check) in enumerate(TimeSeriesSplit(n_splits=5).split(X_dev)):
    print(f"fold {fold + 1}: learn {X_dev['time'].iloc[learn[0]]:%b %d, %Y} to {X_dev['time'].iloc[learn[-1]]:%b %d, %Y}"
          f" | check {X_dev['time'].iloc[check[0]]:%b %d, %Y} to {X_dev['time'].iloc[check[-1]]:%b %d, %Y}")

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/12/images/rolling_window.gif" width="780" alt="Animation of TimeSeriesSplit on the bike data. Top: bikes rented per day, 2011 to 2012. In each frame, a blue stretch the forest learns from grows by 90 days, and the orange 90 days right after it are forecast and checked. Bottom: one lane per fold fills in, with each fold's MAE: 55, 30, 23, 31, 42. The last frame learns everything before July 2012 and checks July to December 2012: MAE 33.">

Each fold is a forecast the bike company could really have made on that date, knowing only what it knew then. The last frame is Part 2's test: the same rolling window, one step further.

> 🎯 **Predict First:** Which folds will promise the smaller error: shuffled or rolling? Which will change more from fold to fold?

In [ ]:
features = clock_weather + lags
folds = {"shuffled (Week 7)": KFold(n_splits=5, shuffle=True, random_state=1),
         "rolling, growing window": TimeSeriesSplit(n_splits=5),
         "rolling, last 6 months only": TimeSeriesSplit(n_splits=5, max_train_size=24 * 182)}
scores = pd.DataFrame({name: -cross_val_score(forest, X_dev[features], y_dev, cv=cv, scoring="neg_mean_absolute_error")
                       for name, cv in folds.items()}, index=["fold 1", "fold 2", "fold 3", "fold 4", "fold 5"]).T
scores["mean"] = scores.mean(axis=1)
scores.round().astype(int)          # MAE, bikes per hour

> 📄 **Handout 4 — Replay the past.** Draw folds 2 to 5. What had fold 1's forest never seen? 🗣️ In June 2012, what do you tell the bike company?

<details><summary>Answer</summary>

* Each fold learns from January 8, 2011 up to its date, then checks the next 90 days. The learning stretch grows: 3, 6, 9, 12, 15 months
* **Fold 1 misses most (55):** it learned from winter only: no warm days, no spring crowds. Fold 2 checks summer, as busy as spring, and misses by 30. Fold 3 (fall, 23) checks quieter months, so its misses are smaller in bikes; fold 5 (spring 2012, 42) checks the busiest ones
* **Shuffled folds: 22, five times.** Each checked hour has learned hours on both sides of it: 7 am and 9 am of the same day, in the same weather. The forest recognizes the day and copies its level: the future leaks in, like scoring on training rows (Week 6). Every fold is the same mix, so all five agree on a number that's too good
* **Rolling folds: 23 to 55, mean 36.** The next six months gave 33. Tell the company: "typically 30 to 40 bikes an hour off; worse in a season unlike any we've seen"
* **Last 6 months only: 37**, a bit worse than the growing window (36). Here old seasons still help: last summer teaches this summer. A fixed window wins when old data stops looking like today (new prices, new rules, a new system)
* Use `cv=TimeSeriesSplit(...)` in `cross_val_score` and in `GridSearchCV`, so knobs are chosen on honest folds too
</details>

From Thomas Aquinas's distinction between *studiositas* (virtuous curiosity) and *curiositas* (vicious curiosity), W. Bradford Littlejohn lists seven vices of curiosity. One of them:

> **Impertinence:** "seeking to know things in a more certain way than one can know, doing violence to the object of knowledge."

> 💬 **Question:** The shuffled folds gave one confident number, five times. Where do you see predictions reported with more certainty than the data can give? What would an honest report to the bike company look like?

<details><summary>Answer</summary>

* The shuffled folds answered an easier question than the company's: they knew the hours around each one. Honest: the rolling folds' range (23 to 55), the seasons where it misses most, and an update as new hours come in
* Elections, markets, sports, "the model says" in the news: a number with no error bar invites people to treat it as fact
* Virtuous curiosity is moved by love, here, for riders who need a bike at 8 am. It plans ahead, and it knows its limits: "you do not know what tomorrow will bring" (James 4:14)
</details>

---
## 5 · A Toolbox for Time-Series Data: Which Tool, When? (SLO 12B) · ~5 min

Every option from today, plus a few we didn't run. Row 1 happened before today: UCI's original file has **no row** for 165 hours (an hour with no rentals got no row). There, `shift(24)` would mean 24 *rows* back, not 24 hours, so `bikes.csv` adds those hours back, with 0 rentals.

| | option | why it helps | when to use it | how, in code |
|---|---|---|---|---|
| | ***Features*** | | | |
| 1 | **Check the clock** | lags count rows, not hours: one missing hour shifts every lag after it | before any lag or rolling feature | `df["time"].diff().value_counts()`; add missing hours with `df.set_index("time").asfreq("h")`, then fill them (`fillna(0)`, `ffill()`, `interpolate()`) |
| 2 | **Lag features** | carry the current level and the recent past into each row: what a tree can't extrapolate | when recent values predict the next. Pick lags at the cycles (1 hour, 1 day, 1 week); each lag costs a column and its first rows | `y.shift(k)`, with `k` ≥ 1: never negative |
| 3 | **Rolling features** | summarize a recent window: steadier than a single lag | noisy data, or when the recent level matters more than any one hour | `y.shift(1).rolling(24).mean()`; also `.sum()`, `.min()`, `.max()`, `.std()` |
| 4 | **Clock features** | give the daily, weekly and yearly cycles | any timestamp. Trees take them as plain numbers | `.dt.hour`, `.dt.dayofweek`, `.dt.month` |
| 5 | **The clock on a circle** | 11 pm sits next to midnight, December next to January | kNN and linear models, when neighbors on the clock behave alike (hours, months; not Sunday and Monday for bikes) | `np.sin(2 * np.pi * hour / 24)` and `np.cos(...)`; `/ 7` for days, `/ 12` for months |
| 6 | **The clock as categories** | any shape over the cycle, both commute peaks included | linear models, with plenty of rows for each value. A `weekend` column when only that split matters | `OneHotEncoder()` on `hour` (Week 9) |
| | ***Testing*** | | | |
| 7 | **Naive forecast** | copies the past: the score any model must beat | always, first | `y.shift(168)` as the prediction |
| 8 | **Time split** | tests on the future, as in real use | always: never a random split | `train = df["time"] < cutoff` |
| 9 | **Rolling window CV** | folds that replay the past: learn from before a date, check what comes next. Shows how the error changes over time | choosing features and knobs, in `cross_val_score` and `GridSearchCV`. Report the folds' range, not just their mean | `cv=TimeSeriesSplit(n_splits=5)` |
| 10 | **Fixed window** | learns from the recent past only | when old data stops looking like today (new prices, new rules, a new system). When old seasons still matter, keep the growing window | `TimeSeriesSplit(n_splits=5, max_train_size=...)` |

**How to choose:** 1, 4, 7 and 8 on every time series. Add 2 and 3, and keep what helps, checked with 9 (or 10). For kNN and linear models, put the cycles on a circle (5) or make them categories (6); trees are fine with plain numbers.

**More options:**
* **Forecasting further ahead.** Our lags assumed we know the last hour. To forecast *h* hours ahead, the newest lag must be *h* hours old (`y.shift(h)`): one model per horizon, or one model fed its own predictions as lags (errors pile up). The further ahead, the bigger the miss: report it for each horizon
* A trend a forest can't follow can be predicted as a *change* instead (`y - y.shift(168)`), then added back
* Classical forecasting models (ARIMA, SARIMA, exponential smoothing, in `statsmodels`) model the series itself; recurrent neural networks learn from long sequences. Both are beyond this course

---
### Wrap-up

1. **Time-series data:** rows in time order, each depending on the ones before
2. **Lag** and **rolling** features carry the recent level, which a forest can't extrapolate. Never shift backwards: that leaks the future
3. Put cycles on a **circle** (sine and cosine) for kNN and linear models: 11 pm next to midnight. Trees don't need it
4. **Test in time order:** a time split, and `TimeSeriesSplit`'s rolling window for CV. Shuffled folds peek, and agree on a number that's too good 📌
5. Beat a naive forecast first, and report how the error changes from fold to fold, not one number
6. Part 5's table is the checklist for any time series